# CRAM plans on Stretch (Isaac Sim, real-robot mode)

Drive the Stretch robot in the Isaac Sim apartment through **CRAM** plans: you
state *what* to achieve -- "park the arm", "raise the torso", "navigate to this
pose" -- as designators, and CRAM binds them to concrete robot motions at run
time, picking the Stretch-specific motion mappings and streaming them to the
giskard control server (`stretch_apartment_giskard_server.py`).

**Kernel**: select **CRAM**.

## Start the simulation and giskard server

Two background processes: the Isaac Sim scene and the giskard control server.
`cram_vrb_lab.control.launcher` starts each and waits for its ready marker. Skip this cell if you
already started them; stale instances are killed first so they do not fight over
the topics. Pass `terminal=True` to open each in a `gnome-terminal` window
instead of the background.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))


import os
os.environ["DISPLAY"] = ":0"

from launcher import start_isaac_sim, start_giskard_server, start_rviz, stop

rviz_proc = start_rviz()          # terminal=True to open a gnome-terminal window
sim_proc = start_isaac_sim(camera="both")          # terminal=True to open a gnome-terminal window
giskard_proc = start_giskard_server()

## Connect and build a CRAM `Context`

CRAM plans run against a `Context`: a world, the robot in it, a ROS node, and the
set of robot-specific motion mappings. We fetch the world the giskard server
built (robot + apartment) over its `fetch_world` service and keep it live with a
`WorldSynchronizer`, then wrap it in a `Context`.

`alternative_motion_mappings` is the list of Stretch-specific motions; CRAM picks
from it by robot type and execution type, so passing the full set is safe.
`evaluate_conditions=False` skips the action pre/post-condition checks (they need
collision queries that are noisy against the coarse apartment model).

In [ ]:
import threading

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from coraplex.alternative_motion_mappings.stretch_motion_mapping import (
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
)
from semantic_digital_twin.robots.stretch import Stretch
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

STRETCH_MOTION_MAPPINGS = [
    StretchMoveToolCenterPoint,
    StretchMoveSim,
    StretchMoveReal,
    StretchClose,
]

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_demo_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

# The world the giskard server published (Stretch + apartment). 300 s matches
# giskardpy's own client: the server may still be parsing the URDF on first start.
world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Stretch)
robot = robot[0] if robot else Stretch.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=STRETCH_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()`
builds each action's giskard motion and streams it to the running server (which
drives Isaac). `collision_avoidance=True` adds an `ExternalCollisionAvoidance`
goal against the apartment the world carries.

In [ ]:
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import sequential, execute_single


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')

## 1. Body: park the arm and raise the torso

`ParkArmsAction` and `MoveTorsoAction` are intent-level: CRAM looks up Stretch's
parked arm configuration and torso joint from the semantic model and turns each
into a `MoveJointsMotion` -- no joint names or target values in the plan.

In [ ]:
from coraplex.robot_plans.actions.core.robot_body import (
    ParkArmsAction,
    MoveTorsoAction,
)
from coraplex.datastructures.enums import Arms
from semantic_digital_twin.datastructures.definitions import TorsoState

run_plan(sequential([
    ParkArmsAction(Arms.LEFT),
    MoveTorsoAction(TorsoState.HIGH),
], context=context))

## 2. Gripper open / close

`SetGripperAction` maps the semantic `GripperState` to the finger joint targets
(0.109 open / 0.0 closed for Stretch).

In [ ]:
from coraplex.robot_plans.actions.core.robot_body import SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))
# run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.CLOSE), context=context))

## 3. Navigate the base to a pose (with apartment collision avoidance)

`NavigateAction` takes a target `Pose` in the world and CRAM resolves it to a
Stretch base motion (`StretchMoveReal` for REAL execution). With
`collision_avoidance=True` the whole-body QP keeps a margin from the apartment
walls/furniture that the shared world carries.

`world.root` is the `map` frame (identical to the Isaac world frame here), so the
target is in world coordinates. The robot spawns near `(-1.5, 0)`; pick a nearby
free spot and adjust if the robot refuses to move (it may be starting too close
to a collision body).

In [ ]:
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from semantic_digital_twin.spatial_types.spatial_types import Pose
from semantic_digital_twin.spatial_types import Point3

waypoints = [
    [0.0, 2, 0.0],
    [1.8, 2, 0.0],
    [1.8, 0, 0.0]
]

for _waypoint in waypoints:
    target = Pose(Point3.from_iterable(_waypoint), reference_frame=world.root)
    run_plan(execute_single(NavigateAction(target), context=context))

## 4. Annotate the drawer (semantic model)

`OpenAction` / `CloseAction` and grasp planning work on the *semantic* model, not
raw links: CRAM needs to know that `cabinet10_drawer_middle` is a `Drawer` and
`handle_cab10_m` is its `Handle`. The apartment URDF only provides the kinematics
(a prismatic joint), so we add the annotations here -- same pattern as CRAM's own
`coraplex_bullet_world_demo`. `modify_world` broadcasts the change over
`/world_sync`, so the giskard server's copy of the world learns it too.

In [ ]:
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
)

drawer_body = world.get_body_by_name("cabinet10_drawer2")
handle_body = world.get_body_by_name("cabinet10_drawer2_handle")

if not world.get_semantic_annotations_by_type(Drawer):
    with world.modify_world():
        world.add_semantic_annotation_recursively(
            Drawer(root=drawer_body, handle=Handle(root=handle_body))
        )
print("drawer annotated:", drawer_body.name, "with handle", handle_body.name)

## 5. Navigate to the cabinet

The pose is expressed in the **apartment's own root frame**, not `map`: CRAM's
apartment test suite (`test_multi_robot_action_designator.test_open`) navigates
to exactly this spot before opening this drawer, and stating it relative to
`apartment_root` means it stays correct however the apartment is placed (or
re-aligned) in the map.

In [ ]:
from semantic_digital_twin.spatial_types import Quaternion

apartment_root = world.get_body_by_name("apartment_root")

at_cabinet10 = Pose(
    Point3.from_iterable([1.6, 1.9, 0]),
    Quaternion.from_iterable([0, 0, 0.3, 1]),
    reference_frame=apartment_root,
)
run_plan(execute_single(NavigateAction(at_cabinet10), context=context))

## 6. Open the drawer

`OpenAction(handle, arm)` expands to *grasp the handle -> pull along the joint
axis -> release*: CRAM reads the prismatic joint from the world model and giskard
executes an `Open` goal, moving the drawer joint in the shared world while the
arm follows. Watch RViz: the drawer slides out in the digital twin.

.. note: the twin is the source of truth here. Isaac renders the same drawer
with a physics joint, but it only moves on screen if the fingers physically hook
the handle -- with the small USD/URDF offset the grasp is approximate, so expect
the rendered drawer to lag behind the twin. `collision_avoidance=False`: the
gripper must get *close* to the furniture, which is exactly what the avoidance
margin forbids.

In [ ]:
from coraplex.robot_plans.actions.core.container import OpenAction, CloseAction

run_plan(
    execute_single(OpenAction(handle_body, Arms.LEFT), context=context),
    collision_avoidance=False,
)
print("drawer joint:", world.get_connection_by_name("cabinet10_drawer_middle_joint").position)

## 7. Spawn an object into the drawer

To pick something out of the drawer, the object must exist in the world. We merge
CRAM's `milk.stl` as a child of the drawer body (so it rides along with the
drawer) and annotate it as `Milk` -- the same recipe the bullet-world demo uses
for the spoon. The merge propagates to the giskard server over `/world_sync`.

The milk exists **only in the digital twin**: Isaac renders no matching rigid
body, so the gripper will close on air on screen while the twin attaches and
carries the object. Spawning a matching body in `stretch_apartment_sim.py` is the
step that closes this gap.

In [ ]:
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk
from semantic_digital_twin.world_description.connections import FixedConnection
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix

MILK_STL = str(
    REPO
    / "cognitive_robot_abstract_machine"
    / "coraplex"
    / "resources"
    / "objects"
    / "milk.stl"
)

if world.get_semantic_annotations_by_type(Milk):
    print("milk already in the world")
else:
    milk_world = STLParser(MILK_STL).parse()
    with world.modify_world():
        world.merge_world(
            milk_world,
            FixedConnection(
                parent=drawer_body,
                child=milk_world.root,
                parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                    0.0, 0.0, 0.12
                ),
            ),
        )
        world.add_semantic_annotations([Milk(root=world.get_body_by_name("milk.stl"))])
print("milk spawned in", drawer_body.name)

## 8. Take it out: transport the milk to the kitchen island

`TransportAction(object, target_pose, arm)` is the full fetch-and-carry
composite: navigate into reach, grasp, lift, carry, place. One designator -- CRAM
resolves every step against the semantic world. The target is the kitchen island
counter, again in apartment coordinates (the bullet-world demo places it at the
same spot).

In [ ]:
from coraplex.robot_plans.actions.composite.transporting import TransportAction

milk_body = world.get_body_by_name("milk.stl")
island_counter = Pose.from_xyz_rpy(
    4.9, 3.3, 0.8, yaw=1.57, reference_frame=apartment_root
)

run_plan(
    execute_single(TransportAction(milk_body, island_counter, Arms.LEFT), context=context),
    collision_avoidance=False,
)

## 9. Close the drawer

Mirror of section 6: navigate back (the transport moved the base away) and run
`CloseAction` on the same handle.

In [ ]:
run_plan(execute_single(NavigateAction(at_cabinet10), context=context))
run_plan(
    execute_single(CloseAction(handle_body, Arms.LEFT), context=context),
    collision_avoidance=False,
)
print("drawer joint:", world.get_connection_by_name("cabinet10_drawer_middle_joint").position)

## Shutdown

Stop the server and the simulation (only if they were started from this notebook).

In [ ]:
stop()  # stops the isaac sim + giskard server started above